# Confronto Risultati CFD — LS59 e Presa a Doppia Rampa
**Corso di Fluidodinamica Computazionale dei Sistemi Propulsivi — Politecnico di Torino**  
**Prof. Andrea Ferrero**

---

Questo notebook confronta i risultati del solutore Eulero 2D (Fortran), di ANSYS Fluent (caso viscoso con Spalart–Allmaras) e i dati sperimentali per due casi applicativi:

| Caso | Quantità confrontata | Motivazione |
|------|---------------------|-------------|
| LS59 (paletta turbina) | Mach isentropico a parete $M_{is}(x'/c)$ | Flusso transsonico prevalentemente isentropico: $M_{is}$ è la grandezza sperimentale di riferimento |
| Presa a doppia rampa | Rapporto di pressione $p_w/p^\circ_\infty$ | Flusso supersonico con urti: $p_{tot}$ cala attraverso gli urti, $M_{is}$ non è più una misura significativa |

### Perché $M_{is}$ per la paletta e $p_w/p^\circ$ per la rampa?

Il **Mach isentropico** si ricava invertendo la relazione isentropica tra pressione statica e totale:
$$M_{is} = \sqrt{\frac{2}{\gamma-1}\left[\left(\frac{p^\circ_\infty}{p_w}\right)^{\frac{\gamma-1}{\gamma}} - 1\right]}$$
Questa formula è fisicamente valida **solo se la pressione totale di freestream $p^\circ_\infty$ rimane costante** lungo tutto il percorso dal freestream alla parete, ovvero solo in assenza di urti o strati limite significativi (flusso isentropico o quasi-isentropico). Per la paletta LS59, che opera in regime transonico con un solutore inviscido (Eulero), questa condizione è soddisfatta. Per la presa a doppia rampa, dove si formano due urti obliqui forti ($M_\infty = 3$), la pressione totale cala irreversibilmente attraverso ogni urto: usare $M_{is}$ darebbe un valore artificialmente distorto. In questo caso si confronta direttamente $p_w/p^\circ_\infty$, che è anche la grandezza misurata sperimentalmente.

### Sistema di riferimento

La paletta LS59 è installata nel canale con un **angolo di calettamento** $\beta_s = 33.3°$ rispetto all'asse globale $x$. Il solutore Eulero lavora nel **sistema di riferimento globale** $(x, y)$; i dati sperimentali sono invece espressi nella coordinata adimensionale **locale** $x'/c$, dove $x'$ è la proiezione sull'asse della paletta e $c = 1/\cos(\beta_s)$ è la lunghezza di corda nel piano globale. La trasformazione richiesta è:
$$x' = x\cos\beta_s - y\sin\beta_s \qquad \Rightarrow \qquad \frac{x'}{c} = (x\cos\beta_s - y\sin\beta_s)\cdot\cos\beta_s$$
Senza questa rotazione, la coordinata $x$ del solutore non è confrontabile con i dati sperimentali.

## 1. Importazione librerie e configurazione

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.interpolate import interp1d
from pathlib import Path

# ---------------------------------------------------------------------------
# CONFIGURAZIONE GRAFICA
# Stile personalizzato: sfondo bianco con griglia leggera, palette cromatica
# differente dal progetto di riferimento (che usava giallo/verde solidi).
# Qui si usa una palette sobria con marker distinti per massima leggibilità
# anche in stampa bianco/nero.
# ---------------------------------------------------------------------------
plt.rcParams.update({
    'figure.figsize'     : (13, 5),
    'axes.grid'          : True,
    'grid.alpha'         : 0.35,
    'grid.linestyle'     : '--',
    'font.size'          : 11,
    'axes.spines.top'    : False,
    'axes.spines.right'  : False,
    'lines.linewidth'    : 1.6,
})

# Palette colori: distinguibile anche in bianco/nero e per daltonici
C_EXP    = '#1a1a2e'   # blu notte  — dati sperimentali (marker ^)
C_EULER  = '#e63946'   # rosso vivo — solutore Eulero
C_FLUENT = '#2a9d8f'   # verde acqua — ANSYS Fluent viscoso

# ---------------------------------------------------------------------------
# PERCORSI
# Tutti i file di input si trovano in SP/Postprocessing/.
# Le figure vengono salvate in SP/Images/.
# ---------------------------------------------------------------------------
DIR_POST   = Path(r"C:\Users\gabri\Desktop\SP\Postprocessing")
DIR_IMAGES = Path(r"C:\Users\gabri\Desktop\SP\Images")
DIR_PALETTA      = Path(r"C:\Users\gabri\Desktop\SP\Paletta")
DIR_DOPPIA_RAMPA = Path(r"C:\Users\gabri\Desktop\SP\Rampa")

# Crea la cartella Images se non esiste
DIR_IMAGES.mkdir(parents=True, exist_ok=True)

# Costante fisica
GAMMA = 1.4

print(f"DIR_POST    : {DIR_POST}")
print(f"DIR_IMAGES  : {DIR_IMAGES}")
print(f"DIR_PALETTA : {DIR_PALETTA}")
print(f"DIR_RAMPA   : {DIR_DOPPIA_RAMPA}")

## 2. Funzioni fisiche e di sanity check

In [ ]:
def calcola_mach_isentropico(p_stat, p_tot_inf, gamma=GAMMA):
    """
    Calcola il Mach isentropico a parete.

    Formula:
        M_is = sqrt( 2/(gamma-1) * [ (p0_inf/p_w)^((gamma-1)/gamma) - 1 ] )

    ARGOMENTI
    ---------
    p_stat    : array, pressione statica a parete (stessa unità di p_tot_inf)
    p_tot_inf : scalare, pressione totale di FREESTREAM
                - Per il solutore Eulero: p0_inf = 1.0 (adimensionale, BC inlet)
                - Per Fluent: p0_inf = max(p_tot a parete) [Pa], che corrisponde
                  al valore di freestream indisturbato (la p_tot non può aumentare)
    gamma     : rapporto calori specifici (default 1.4 per aria)

    NOTA CRITICA: p_tot_inf DEVE essere uno scalare costante.
    Usare la pressione totale locale (variabile lungo x) introduce un errore
    fisico: nelle zone di strato limite o post-urto p_tot cala, e il Mach
    calcolato risulterebbe artificialmente distorto.
    """
    rapporto = np.clip(p_tot_inf / np.asarray(p_stat, dtype=float), 1.0, None)
    return np.sqrt((2.0 / (gamma - 1.0)) * (rapporto ** ((gamma - 1.0) / gamma) - 1.0))


def sanity_check_mach(mach, nome):
    """Verifica che M >= 0 e che non ci siano NaN/Inf."""
    ok = True
    if np.any(np.isnan(mach)) or np.any(np.isinf(mach)):
        print(f"  [ERRORE] NaN/Inf nel Mach — {nome}")
        ok = False
    if np.any(mach < 0):
        print(f"  [ERRORE] Valori M < 0 — {nome}")
        ok = False
    if ok:
        print(f"  [OK] Mach — {nome:35s}  M_max = {np.max(mach):.4f}")


def sanity_check_pressione(p_ratio, nome):
    """Verifica che 0 <= p/p0 <= 1 (con tolleranza numerica 1%)."""
    ok = True
    if np.any(np.isnan(p_ratio)):
        print(f"  [ERRORE] NaN nel rapporto di pressione — {nome}")
        ok = False
    if np.any(p_ratio < 0) or np.any(p_ratio > 1.01):
        print(f"  [WARNING] p/p0 fuori [0,1] — {nome}  (max={np.max(p_ratio):.4f})")
        ok = False
    if ok:
        print(f"  [OK] p/p0   — {nome:35s}  max = {np.max(p_ratio):.4f}")


def sanity_check_x(x, nome, x_min=-0.1, x_max=1.1):
    """Verifica che la coordinata adimensionale x'/c cada nel range atteso."""
    if x.min() < x_min or x.max() > x_max:
        print(f"  [WARNING] x fuori range [{x_min},{x_max}] — {nome}  "
              f"(got [{x.min():.3f}, {x.max():.3f}])")
    else:
        print(f"  [OK] x/c    — {nome:35s}  range [{x.min():.3f}, {x.max():.3f}]")

## 3. Funzioni di importazione dati

In [ ]:
def import_eulero_wall(filepath):
    """
    Importa i file di output del solutore Eulero 2D (formato testo, no header).

    STRUTTURA ATTESA: 3 colonne
        col 0 : x  (coordinata globale, adimensionale)
        col 1 : y  (coordinata globale, adimensionale)
        col 2 : p/p0_inf  (rapporto pressione statica / pressione totale freestream)
                          Nota: il solutore impone p0_inf = 1 come BC di inlet,
                          quindi col2 = p_stat direttamente.

    Il file contiene i punti di TUTTE le pareti (upper + lower) mescolati;
    la separazione viene eseguita a valle in base alla coordinata y.
    """
    data = np.loadtxt(filepath)
    if data.shape[1] < 3:
        raise ValueError(f"File {filepath}: attese almeno 3 colonne, trovate {data.shape[1]}")
    return data  # restituisce l'array grezzo (x, y, p/p0)


def import_fluent_xy(filepath):
    """
    Importa i file .xy esportati da ANSYS Fluent (XY Plot Export).

    STRATEGIA DI PARSING:
    Tenta la conversione float su ogni coppia di token: se riesce è un dato,
    altrimenti la riga è header LISP/parentesi e viene ignorata silenziosamente.
    Questo gestisce correttamente:
    - Header del tipo: (title ...), ((xy/key/label ...))
    - Righe di chiusura ')' che in file multi-zona appaiono in mezzo al file
    """
    x_list, y_list = [], []
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.split()
            if len(parts) == 2:
                try:
                    x_list.append(float(parts[0]))
                    y_list.append(float(parts[1]))
                except ValueError:
                    pass
    if not x_list:
        raise ValueError(f"Nessun dato numerico trovato in: {filepath}")
    data = np.column_stack([x_list, y_list])
    data = data[data[:, 0].argsort()]  # ordina per x crescente
    return data[:, 0], data[:, 1]


def import_sperimentale(filepath):
    """
    Importa dati sperimentali con header testuale (separatore spazio/tab).
    Colonna 0 = x adimensionale, colonna 1 = grandezza misurata.
    """
    df = pd.read_csv(filepath, sep=r'\s+', header=0)
    return df.iloc[:, 0].values, df.iloc[:, 1].values


def import_sperimentale_noheader(filepath):
    """
    Importa dati sperimentali senza header (due colonne numeriche).
    Usato per i file del progetto di riferimento (anno precedente).
    """
    data = np.loadtxt(filepath)
    data = data[data[:, 0].argsort()]
    return data[:, 0], data[:, 1]


def import_fluent_wall_txt(filepath, p_col=1):
    """
    Importa file di parete Fluent in formato testo puro (no header Lisp).
    Formato: col0=x, col_p=pressione statica [Pa].
    Usato per i file del progetto di riferimento (wall_data_fluent_*.txt).
    """
    data = np.loadtxt(filepath)
    data = data[data[:, 0].argsort()]
    return data[:, 0], data[:, p_col]

---
## 4. Paletta LS59 — Mach Isentropico a Parete

### 4.1 Sistema di coordinate e trasformazione

La paletta LS59 è installata con angolo di calettamento $\beta_s = 33.3°$.
Il solutore Eulero produce coordinate nel **sistema globale** $(x, y)$ in cui la paletta
è ruotata. I dati sperimentali usano la coordinata **locale** $x'/c$ dove $x'$ è la
proiezione lungo la direzione della corda.

La trasformazione è:
$$x' = x\cos\beta_s - y\sin\beta_s$$
$$c_{\text{globale}} = \frac{C_{\text{locale}}}{\cos\beta_s} = \frac{1}{\cos\beta_s}$$
$$\frac{x'}{c} = (x\cos\beta_s - y\sin\beta_s)\cdot\cos\beta_s$$

La **separazione delle superfici** (dorso/ventre) non può essere fatta con una
semplice soglia su $y$ perché la paletta è inclinata. Si usa un fit polinomiale
di 2° grado sulla nuvola di punti $(x, y)$ della parete: i punti sopra il polinomio
appartengono al **dorso** (suction side), quelli sotto al **ventre** (pressure side).

In [ ]:
# ---------------------------------------------------------------------------
# PARAMETRI GEOMETRICI LS59
# ---------------------------------------------------------------------------
BETAS_DEG = 33.3                          # angolo di calettamento [gradi]
BETAS     = BETAS_DEG * np.pi / 180.0    # radianti
CORDA     = 1.0 / np.cos(BETAS)          # lunghezza di corda nel SR globale

print(f"Angolo di calettamento:  beta_s = {BETAS_DEG}°")
print(f"Corda nel SR globale:    c      = {CORDA:.6f}")
print()

# ---------------------------------------------------------------------------
# DATI SPERIMENTALI
# Fonte: exp_data_LS59_wall_Mis_alpha30_Misexit1_2.txt
# Già in coordinata adimensionale x'/c vs M_is sperimentale
# ---------------------------------------------------------------------------
# --- Prova prima i file tuoi, poi fallback ai dati di riferimento ---
try:
    x_exp_pal, mach_exp_pal = import_sperimentale(DIR_PALETTA / 'exp_M_is(x).txt')
    print("[CARICATO] Dati sperimentali: file tuo (exp_M_is(x).txt)")
except FileNotFoundError:
    # Fallback al file del progetto di riferimento (anno precedente)
    ref_file = DIR_POST / 'exp_data_LS59_wall_Mis_alpha30_Misexit1_2.txt'
    x_exp_pal, mach_exp_pal = import_sperimentale_noheader(ref_file)
    print(f"[FALLBACK] Dati sperimentali: file anno precedente ({ref_file.name})")

# ---------------------------------------------------------------------------
# DATI EULERO
# Il file wall_data contiene (x_glob, y_glob, p/p0_inf) per tutti i punti parete.
# p0_inf = 1 (condizione al contorno del solutore), quindi col2 = p_stat.
# ---------------------------------------------------------------------------
try:
    eul_raw_pal = import_eulero_wall(DIR_PALETTA / 'eulero_paletta.txt')
    print("[CARICATO] Euler LS59: file tuo (eulero_paletta.txt)")
except FileNotFoundError:
    eul_raw_pal = import_eulero_wall(DIR_POST / 'wall_data_LS59.txt')
    print("[FALLBACK] Euler LS59: wall_data_LS59.txt (anno precedente)")

# ---------------------------------------------------------------------------
# DATI FLUENT (viscoso Spalart–Allmaras)
# Due opzioni di file:
#   A) File .xy di Fluent (export XY Plot): fluent_pwall(x).txt  [formato LISP]
#   B) File testo dal progetto di riferimento: pressure_x.dat + pressure_y.dat
#      dove la pressione in Pa si normalizza dividendo per p0_fluent = 100000 Pa
#
# NOTA: il progetto dell'anno scorso ha sia caso inviscido (pressure_x.dat)
# che viscoso Spalart-Allmaras (pressure_x_spalat.dat). Lo schema della relazione
# di quest'anno richiede solo il confronto con Fluent (caso viscoso con S-A),
# quindi usiamo pressure_x_spalat.dat come riferimento se non disponibile il nostro.
# ---------------------------------------------------------------------------
P0_FLUENT_PA = 100000.0  # pressione totale di freestream in Fluent [Pa]

try:
    x_p_pal,    p_stat_pa_pal  = import_fluent_xy(DIR_PALETTA / 'fluent_pwall(x).txt')
    x_ptot_pal, p_tot_pa_pal   = import_fluent_xy(DIR_PALETTA / 'fluent_pwall_tot(x).txt')
    p_tot_inf_pal = float(np.max(p_tot_pa_pal))   # freestream = massimo della p_tot
    print(f"[CARICATO] Fluent LS59: file tuoi | p0_inf = {p_tot_inf_pal:.1f} Pa")
    fonte_fluent_pal = 'file corrente'
except FileNotFoundError:
    # Fallback: pressure_x_spalat.dat (x, p[Pa]) + pressure_y_spalat.dat (y, p[Pa])
    # Stesso formato: col0=coordinata spaziale, col1=pressione[Pa]
    px_s = np.loadtxt(DIR_POST / 'pressure_x_spalat.dat')
    py_s = np.loadtxt(DIR_POST / 'pressure_y_spalat.dat')
    # x e y sono sulle stesse celle -> assembliamo (x_glob, y_glob, p[Pa])
    x_glob_flu_pal = px_s[:, 0]
    y_glob_flu_pal = py_s[:, 0]
    p_stat_pa_pal_raw = px_s[:, 1]   # pressione [Pa]
    p_tot_inf_pal = P0_FLUENT_PA
    # Costruiamo x_p_pal come x_rot/corda per consistenza con Euler
    x_p_pal = (np.cos(BETAS) * x_glob_flu_pal - np.sin(BETAS) * y_glob_flu_pal) / CORDA
    p_stat_pa_pal = p_stat_pa_pal_raw
    print(f"[FALLBACK] Fluent LS59: pressure_x_spalat.dat | p0_inf = {p_tot_inf_pal:.0f} Pa")
    fonte_fluent_pal = 'fallback (anno precedente - Spalart-Allmaras)'

print(f"  Fonte Fluent LS59: {fonte_fluent_pal}")

In [ ]:
# ---------------------------------------------------------------------------
# SEPARAZIONE DORSO / VENTRE — Eulero LS59
#
# La paletta è inclinata: non è possibile separare le due superfici con una
# semplice soglia su y. Si usa un polinomio di 2° grado fitting la nuvola
# di punti (x, y): i punti sopra la curva = dorso, quelli sotto = ventre.
# Questo è il metodo usato nel progetto di riferimento (anno precedente).
# ---------------------------------------------------------------------------
x_eul = eul_raw_pal[:, 0]
y_eul = eul_raw_pal[:, 1]
p_eul = eul_raw_pal[:, 2]   # p_stat / p0_inf  (p0_inf=1)

# Fit parabolico su un sottoinsieme decimato per evitare overfitting locale
step_fit = max(1, len(x_eul) // 50)
coeff = np.polyfit(x_eul[::step_fit], y_eul[::step_fit], 2)
y_parab = np.polyval(coeff, x_eul)

mask_dorso  = y_eul >= y_parab
mask_ventre = y_eul <  y_parab

# Coordinata adimensionale locale x'/c per entrambe le superfici
x_rot_eul = np.cos(BETAS) * x_eul - np.sin(BETAS) * y_eul
xc_eul_dorso  = x_rot_eul[mask_dorso]  / CORDA
xc_eul_ventre = x_rot_eul[mask_ventre] / CORDA

# Mach isentropico Eulero (p0_inf = 1 -> uso scalare 1.0)
mach_eul_dorso  = calcola_mach_isentropico(p_eul[mask_dorso],  1.0)
mach_eul_ventre = calcola_mach_isentropico(p_eul[mask_ventre], 1.0)

# ---------------------------------------------------------------------------
# MACH ISENTROPICO FLUENT
# p_tot_inf_pal è il valore scalare estratto sopra (max p_tot o 100000 Pa)
# ---------------------------------------------------------------------------
mach_flu_pal = calcola_mach_isentropico(p_stat_pa_pal, p_tot_inf_pal)

# Per il file di fallback, x_p_pal è già in x'/c; per il file corrente bisogna
# verificare se è già coordinata locale o globale:
# I file .xy di Fluent esportati lungo 'wall' usano la coordinata curvilinea
# o la coordinata x del SR globale. Aggiungiamo la rotazione se necessario.
# (Commentare/decommentare in base al formato del proprio file Fluent)
# x_p_pal = (np.cos(BETAS) * x_p_pal_globale - np.sin(BETAS) * y_p_pal_globale) / CORDA

# ---------------------------------------------------------------------------
# SANITY CHECKS
# ---------------------------------------------------------------------------
print("=== Sanity Check — LS59 ===")
sanity_check_mach(mach_exp_pal,    "Sperimentale")
sanity_check_mach(mach_eul_dorso,  "Eulero - Dorso")
sanity_check_mach(mach_eul_ventre, "Eulero - Ventre")
sanity_check_mach(mach_flu_pal,    "Fluent (Spalart–Allmaras)")
print()
sanity_check_x(x_exp_pal,      "Sperimentale")
sanity_check_x(xc_eul_dorso,   "Eulero - Dorso")
sanity_check_x(xc_eul_ventre,  "Eulero - Ventre")
sanity_check_x(x_p_pal,        "Fluent")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(r"$M_{is}$ — Paletta LS59", fontsize=13, fontweight='bold')

# ---------- Plot 1: Exp vs Eulero ----------
ax = axes[0]
ax.plot(x_exp_pal, mach_exp_pal,
        marker='^', ms=7, ls='none', color=C_EXP,
        label='Sperimentale', zorder=5)
ax.plot(xc_eul_dorso[::3], mach_eul_dorso[::3],
        marker='o', ms=3, ls='-', color=C_EULER, lw=1.2,
        label='Eulero — Dorso', zorder=3)
ax.plot(xc_eul_ventre[::3], mach_eul_ventre[::3],
        marker='s', ms=3, ls='--', color=C_EULER, lw=1.2,
        label='Eulero — Ventre', zorder=3)
ax.set_xlabel(r"$x'/c$")
ax.set_ylabel(r"$M_{is}$")
ax.set_title("Sperimentale vs Eulero")
ax.legend(fontsize=9)

# ---------- Plot 2: Confronto globale ----------
ax2 = axes[1]
ax2.plot(x_exp_pal, mach_exp_pal,
         marker='^', ms=7, ls='none', color=C_EXP,
         label='Sperimentale', zorder=5)
ax2.plot(xc_eul_dorso[::3], mach_eul_dorso[::3],
         marker='o', ms=3, ls='-', color=C_EULER, lw=1.2,
         label='Eulero — Dorso', zorder=3)
ax2.plot(xc_eul_ventre[::3], mach_eul_ventre[::3],
         marker='s', ms=3, ls='--', color=C_EULER, lw=1.2,
         label='Eulero — Ventre', zorder=3)
ax2.plot(x_p_pal[::2], mach_flu_pal[::2],
         marker='none', ls='-', color=C_FLUENT, lw=2.0,
         label='Fluent (Spalart–Allmaras)', zorder=4)
ax2.set_xlabel(r"$x'/c$")
ax2.set_ylabel(r"$M_{is}$")
ax2.set_title("Confronto Globale")
ax2.legend(fontsize=9)

plt.tight_layout()
out_path = DIR_IMAGES / 'confronto_Mis_LS59.png'
plt.savefig(out_path, dpi=180, bbox_inches='tight')
plt.show()
print(f"Figura salvata: {out_path}")

---
## 5. Presa a Doppia Rampa — Rapporto di Pressione $p_w / p^\circ_\infty$

### 5.1 Perché non si usa $M_{is}$ qui?

Nella presa a doppia rampa il flusso in ingresso è supersonico ($M_\infty = 3$) e
attraversa due urti obliqui. Attraverso ogni urto la **pressione totale cala
irreversibilmente** (seconda legge della termodinamica: produzione di entropia).
Se si usasse la formula isentropica con $p^\circ_\infty$ costante, si otterrebbe un
$M_{is}$ che non riflette il Mach reale locale ma solo la "distanza" dallo stato di
freestream — una grandezza non significativa per un confronto fisico post-urto.
La grandezza giusta è il rapporto diretto $p_w/p^\circ_\infty$, che è anche quella
misurata sperimentalmente.

### 5.2 Selezione della parete inferiore

Il file di output del solutore contiene i punti di tutte le pareti.
La doppia rampa è la **parete inferiore** del canale: si seleziona con il filtro $y < 0.25$
(la parete superiore è a $y \approx 0.8$).

In [ ]:
# ---------------------------------------------------------------------------
# DATI SPERIMENTALI — Doppia Rampa
# Grandezza: p_w / p0_inf  (già adimensionale)
# ---------------------------------------------------------------------------
try:
    x_exp_dr, pr_exp_dr = import_sperimentale(
        DIR_DOPPIA_RAMPA / 'exp_p_wall_p_tot_inf(x).txt')
    print("[CARICATO] Dati sperimentali Rampa: file tuo")
except FileNotFoundError:
    x_exp_dr, pr_exp_dr = import_sperimentale_noheader(
        DIR_POST / 'exp_data_double_wedge_inlet_Mach3.txt')
    print("[FALLBACK] Dati sperimentali Rampa: exp_data_double_wedge_inlet_Mach3.txt")

# ---------------------------------------------------------------------------
# DATI EULERO — Doppia Rampa
# Colonne: (x_glob, y_glob, p/p0_inf)
# Filtriamo la parete inferiore: y < 0.25
# ---------------------------------------------------------------------------
try:
    eul_raw_dr = import_eulero_wall(DIR_DOPPIA_RAMPA / 'eulero_p_wall_p_tot.txt')
    print("[CARICATO] Eulero Rampa: file tuo")
except FileNotFoundError:
    eul_raw_dr = import_eulero_wall(DIR_POST / 'wall_data_rampa.txt')
    print("[FALLBACK] Eulero Rampa: wall_data_rampa.txt (anno precedente)")

# Filtro parete inferiore
Y_THRESHOLD_RAMPA = 0.25
mask_lower = eul_raw_dr[:, 1] < Y_THRESHOLD_RAMPA
x_eul_dr  = eul_raw_dr[mask_lower, 0]
pr_eul_dr = eul_raw_dr[mask_lower, 2]   # già p/p0_inf

# Ordina per x crescente
ord_idx = np.argsort(x_eul_dr)
x_eul_dr  = x_eul_dr[ord_idx]
pr_eul_dr = pr_eul_dr[ord_idx]

# ---------------------------------------------------------------------------
# DATI FLUENT — Doppia Rampa
# ---------------------------------------------------------------------------
try:
    x_p_dr,    p_stat_pa_dr  = import_fluent_xy(DIR_DOPPIA_RAMPA / 'fluent_p(x).txt')
    x_ptot_dr, p_tot_pa_dr   = import_fluent_xy(DIR_DOPPIA_RAMPA / 'fluent_p_tot(x).txt')
    p_tot_inf_dr = float(np.max(p_tot_pa_dr))   # freestream = massimo
    print(f"[CARICATO] Fluent Rampa: file tuoi | p0_inf = {p_tot_inf_dr:.1f} Pa")
except FileNotFoundError:
    # Fallback: wall_data_fluent_visc.txt (x, p[Pa])
    x_flu_raw, p_flu_raw = import_fluent_wall_txt(DIR_POST / 'wall_data_fluent_visc.txt')
    p_tot_inf_dr = P0_FLUENT_PA
    x_p_dr       = x_flu_raw
    p_stat_pa_dr = p_flu_raw
    print(f"[FALLBACK] Fluent Rampa: wall_data_fluent_visc.txt | p0_inf = {p_tot_inf_dr:.0f} Pa")

pr_flu_dr = p_stat_pa_dr / p_tot_inf_dr

# ---------------------------------------------------------------------------
# SANITY CHECKS
# ---------------------------------------------------------------------------
print()
print("=== Sanity Check — Doppia Rampa ===")
sanity_check_pressione(pr_exp_dr,  "Sperimentale")
sanity_check_pressione(pr_eul_dr,  "Eulero (parete inferiore)")
sanity_check_pressione(pr_flu_dr,  "Fluent (Spalart–Allmaras)")
print()
print(f"Punti Eulero su parete inferiore (y < {Y_THRESHOLD_RAMPA}): {mask_lower.sum()} / {len(eul_raw_dr)}")
print(f"x Eulero rampa: [{x_eul_dr.min():.3f}, {x_eul_dr.max():.3f}]")
print(f"x Fluent rampa: [{x_p_dr.min():.3f}, {x_p_dr.max():.3f}]")
print(f"x Exp rampa:    [{x_exp_dr.min():.3f}, {x_exp_dr.max():.3f}]")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(r"$p_w / p^\circ_\infty$ — Presa a Doppia Rampa ($M_\infty = 3$)",
             fontsize=13, fontweight='bold')

# ---------- Plot 1: Exp vs Eulero ----------
ax = axes[0]
ax.plot(x_exp_dr, pr_exp_dr,
        marker='^', ms=7, ls='none', color=C_EXP,
        label='Sperimentale', zorder=5)
ax.plot(x_eul_dr, pr_eul_dr,
        marker='o', ms=3, ls='-', color=C_EULER, lw=1.4,
        label='Eulero 2D (Roe)', zorder=3)
ax.set_xlabel(r"$x/L$")
ax.set_ylabel(r"$p_w / p^\circ_\infty$")
ax.set_title("Sperimentale vs Eulero")
ax.legend(fontsize=9)

# ---------- Plot 2: Confronto globale ----------
ax2 = axes[1]
ax2.plot(x_exp_dr, pr_exp_dr,
         marker='^', ms=7, ls='none', color=C_EXP,
         label='Sperimentale', zorder=5)
ax2.plot(x_eul_dr, pr_eul_dr,
         marker='o', ms=3, ls='-', color=C_EULER, lw=1.4,
         label='Eulero 2D (Roe)', zorder=3)
ax2.plot(x_p_dr, pr_flu_dr,
         marker='none', ls='-', color=C_FLUENT, lw=2.0,
         label='Fluent (Spalart–Allmaras)', zorder=4)
ax2.set_xlabel(r"$x/L$")
ax2.set_ylabel(r"$p_w / p^\circ_\infty$")
ax2.set_title("Confronto Globale")
ax2.legend(fontsize=9)

plt.tight_layout()
out_path = DIR_IMAGES / 'confronto_pressione_doppia_rampa.png'
plt.savefig(out_path, dpi=180, bbox_inches='tight')
plt.show()
print(f"Figura salvata: {out_path}")

---
## 6. Cella di debug — verifica $p^\circ_\infty$ estratto da Fluent

Prima di procedere con i plot principali, verificare visivamente che il valore di $p^\circ_\infty$
estratto corrisponda al plateau del freestream indisturbato (zona di ingresso del dominio).
Se la linea tratteggiata coincide con il tratto iniziale costante, l'estrazione è corretta.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("Debug: andamento $p_{tot}$ a parete da Fluent", fontsize=12)

for ax, title, x_ptot, p_tot_arr, p_inf in [
    (axes[0], "LS59",
     x_ptot_pal if 'x_ptot_pal' in dir() else x_p_pal,
     p_tot_pa_pal if 'p_tot_pa_pal' in dir() else p_stat_pa_pal * 0 + p_tot_inf_pal,
     p_tot_inf_pal),
    (axes[1], "Doppia Rampa",
     x_ptot_dr if 'x_ptot_dr' in dir() else x_p_dr,
     p_tot_pa_dr if 'p_tot_pa_dr' in dir() else p_stat_pa_dr * 0 + p_tot_inf_dr,
     p_tot_inf_dr),
]:
    ax.plot(x_ptot, p_tot_arr, color='#e76f51', lw=1.5)
    ax.axhline(p_inf, color='#264653', ls='--', lw=1.5,
               label=f'$p^\circ_\infty$ estratto = {p_inf:.1f} Pa')
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('$p_{tot}$ [Pa]')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(DIR_IMAGES / 'debug_ptot_freestream.png', dpi=150, bbox_inches='tight')
plt.show()
print()
print("Se la linea tratteggiata NON coincide con il plateau iniziale:")
print("  Sovrascrivi manualmente, es.:")
print("  p_tot_inf_pal = 99500.0  # valore letto dal grafico")
print("  p_tot_inf_dr  = 99800.0")